In [1]:
"""
CineMatch — XSimGCL Collaborative Filtering via RecBole
========================================================
Trains an XSimGCL (Cross-batch Simulated Graph Contrastive Learning) model
on the MovieLens 32M dataset using RecBole. Exports user/item embeddings
for downstream late fusion.

Features:
  - temporal eval split** before training 
  - Automatic conversion of MovieLens CSV → RecBole .inter format
  - XSimGCL config with tuned hyperparameters for long-tail items
  - Exports user_embeddings.npy and item_embeddings.npy
  - ID mapping files for MovieLens ↔ RecBole translation
"""


'\nCineMatch — XSimGCL Collaborative Filtering via RecBole\n========================================================\nTrains an XSimGCL (Cross-batch Simulated Graph Contrastive Learning) model\non the MovieLens 32M dataset using RecBole. Exports user/item embeddings\nfor downstream late fusion.\n\nFeatures:\n  - temporal eval split** before training \n  - Automatic conversion of MovieLens CSV → RecBole .inter format\n  - XSimGCL config with tuned hyperparameters for long-tail items\n  - Exports user_embeddings.npy and item_embeddings.npy\n  - ID mapping files for MovieLens ↔ RecBole translation\n'

In [2]:
# # !pip install recbole==1.1.1
# # !pip install torch-geometric
# # !git clone https://github.com/RUCAIBox/RecBole-GNN.git
# import sys
# sys.path.insert(0, "/content/RecBole-GNN")

In [3]:
# !pip uninstall torch-scatter torch-sparse torch-geometric torch-cluster  --y
# !pip install torch-sparse -f https://data.pyg.org/whl/torch-{torch.__version__}.html
# !pip install torch-cluster -f https://data.pyg.org/whl/torch-{torch.__version__}.html
# !pip install git+https://github.com/pyg-team/pytorch_geometric.git

In [4]:
import sys
sys.path.insert(0, "/blue/egn6933/nagabhairava.r/RecBole-GNN")

In [5]:
from __future__ import annotations

import os
import warnings
os.environ['PYTHONWARNINGS'] = 'ignore'
warnings.filterwarnings('ignore')

import torch
if not hasattr(torch, "_original_load"):
    torch._original_load = torch.load
    def safe_load(f, map_location=None, pickle_module=None, **kwargs):
        kwargs.setdefault("weights_only", False)
        return torch._original_load(f, map_location=map_location, **kwargs)
    torch.load = safe_load

import sys
import time
import json
import random
import numpy as np
import pandas as pd
from dataclasses import dataclass, asdict
from pathlib import Path


# CONFIG


# XSimGCL Hyperparameters
EMBEDDING_SIZE = 512      # dimensionality of user/item embeddings
N_LAYERS       = 3         # number of GCN layers
CL_RATE        = 0.5       # contrastive learning loss weight
NOISE_EPS      = 0.1       # noise perturbation epsilon for SimGCL
REG_WEIGHT     = 1e-4      # L2 regularization

# Training
LEARNING_RATE  = 1e-3
TRAIN_BATCH    = 262144
EPOCHS         = 50     # max epochs
EARLY_STOP     = 5        # patience
EVAL_BATCH     = 70000000

# Data
RATING_THRESHOLD = 3.5     # implicit positive threshold
TEMPORAL_SPLIT   = True    # True = temporal split, False = RecBole random
SPLIT_RATIO      = [0.8, 0.1, 0.1]  # train/val/test (RecBole internal)

# Evaluation split
N_TEST_USERS       = 1000     # number of test users to hold out
N_HOLDOUT          = 10       # movies to hide per test user
MIN_USER_RATINGS   = 20       # minimum total ratings to be eligible
HOLDOUT_THRESHOLD  = 4.0      # only hide movies rated >= this
EVAL_SEED          = 42       # fixed seed for reproducible split

# Demographics (for website users)
AGE_BUCKETS = ["18-24", "25-34", "35-44", "45-54", "55+"]
GENDER_OPTIONS = ["M", "F", "undisclosed"]
REGION_OPTIONS = [
    "USA",
    "Canada", "UK", "Europe", "Latin-America",
    "Asia", "Middle-East", "Africa", "Other",
]
DEMO_BLEND_WEIGHT = 0.3
MIN_CLUSTER_SIZE  = 5

@dataclass
class UserProfile:
    """Demographic profile collected from website signup."""
    age_group: str = "undisclosed"
    gender: str    = "undisclosed"
    region: str    = "Other"

    def cluster_keys(self) -> list[tuple]:
        """Return lookup keys from most to least specific for fallback matching."""
        return [
            (self.age_group, self.gender, self.region),
            (self.age_group, self.gender, "*"),
            (self.age_group, "*", "*"),
            ("*", self.gender, "*"),
            ("*", "*", self.region),
        ]


In [6]:
def detect_paths() -> dict:
    try:
        from google.colab import drive     # type: ignore
        drive.mount("/content/drive", force_remount=False)
        base = Path("/content/drive/MyDrive/cinematch/Data")
        print("Runtime: Colab")
    except ImportError:
        hpc = Path("/blue/egn6933/nagabhairava.r")
        if hpc.exists():
            base = hpc
            print("Runtime: HPC")
        else:
            here = Path(__file__).resolve().parent if "__file__" in dir() else Path.cwd()
            for candidate in [here, *here.parents]:
                if (candidate / "Data").exists() and (candidate / "src").exists():
                    base = candidate / "Data"
                    break
            else:
                base = Path.cwd() / "Data"
            print("Runtime: Local")

    model_dir = base / "outputs" / "xsimgcl"
    if not model_dir.is_absolute():
        model_dir = model_dir.resolve()
    model_dir.mkdir(parents=True, exist_ok=True)

    recbole_data = model_dir / "dataset" / "cinematch"
    recbole_data.mkdir(parents=True, exist_ok=True)

    ml_dir = base / "ml-32m"
    if not ml_dir.exists():
        ml_dir = base / "Data" / "ml-32m"

    return {
        "base":            base,
        "ratings_csv":     ml_dir / "ratings.csv",
        "movies_csv":      ml_dir / "movies.csv",
        "links_csv":       ml_dir / "links.csv",
        "model_dir":       model_dir,
        "recbole_data":    recbole_data,
        "inter_file":      recbole_data / "cinematch.inter",
        "user_emb":        model_dir / "user_embeddings.npy",
        "item_emb":        model_dir / "item_embeddings.npy",
        "user_id_map":     model_dir / "user_id_map.json",
        "item_id_map":     model_dir / "item_id_map.json",
        "config_yaml":     model_dir / "xsimgcl_config.yaml",
        "train_manifest":  model_dir / "train_manifest.json",
        # Eval split outputs
        "train_ratings_csv":  model_dir / "train_ratings.csv",
        "test_holdout_csv":   model_dir / "test_holdout.csv",
        "eval_split_meta":    model_dir / "eval_split_meta.json",
        # Demographic files
        "demo_profiles":     model_dir / "user_demographics.csv",
        "demo_clusters":     model_dir / "demographic_clusters.npy",
        "demo_cluster_map":  model_dir / "demographic_cluster_map.json",
    }


In [ ]:

def create_eval_split(paths: dict) -> dict:
    """
    Create a strict temporal train/test split BEFORE training.

    For each of N_TEST_USERS eligible users, hide their last N_HOLDOUT
    highly-rated movies (≥ HOLDOUT_THRESHOLD) as ground truth.

    Saves:
      - train_ratings.csv : all ratings minus held-out interactions
      - test_holdout.csv  : held-out interactions (userId, movieId, rating, timestamp)
      - eval_split_meta.json : split statistics

    Returns dict with split info.
    """
    print("Creating evaluation split")
    print(f"{'─'*60}")

    assert paths["ratings_csv"].exists(), f"Missing: {paths['ratings_csv']}"

    # Load full ratings
    dtypes = {"userId": "int32", "movieId": "int32", "rating": "float32", "timestamp": "int32"}
    ratings = pd.read_csv(paths["ratings_csv"], dtype=dtypes)
    print(f"  Loaded {len(ratings):,} total ratings")

    # Set reproducible seed
    random.seed(EVAL_SEED)
    np.random.seed(EVAL_SEED)

    # Find eligible users (those with enough ratings)
    user_counts = ratings.groupby("userId").size()
    eligible = user_counts[user_counts >= MIN_USER_RATINGS].index.tolist()
    print(f"  Eligible users (≥{MIN_USER_RATINGS} ratings): {len(eligible):,}")

    # Stratified sample by activity level
    user_counts_eligible = user_counts[eligible]
    bins = [0, 50, 100, 200, 500, float("inf")]
    labels = ["5-50", "50-100", "100-200", "200-500", "500+"]
    user_bins = pd.cut(user_counts_eligible, bins=bins, labels=labels)

    sampled_users = []
    per_bin = N_TEST_USERS // len(labels)
    for label in labels:
        bin_users = user_bins[user_bins == label].index.tolist()
        n = min(per_bin, len(bin_users))
        sampled_users.extend(random.sample(bin_users, n))
    # Fill remainder randomly
    remaining = [u for u in eligible if u not in set(sampled_users)]
    if len(sampled_users) < N_TEST_USERS:
        sampled_users.extend(random.sample(remaining, N_TEST_USERS - len(sampled_users)))
    sampled_users = sampled_users[:N_TEST_USERS]
    print(f"  Sampled {len(sampled_users)} test users (stratified by activity)")

    # Build held-out set
    holdout_indices = []
    holdout_rows = []

    test_set = set(sampled_users)
    grouped = (
        ratings[ratings["userId"].isin(test_set)]
        .sort_values("timestamp")
        .groupby("userId")
    )

    for uid, group in grouped:
        liked = group[group["rating"] >= HOLDOUT_THRESHOLD]
        if len(liked) < N_HOLDOUT:
            continue
        holdout = liked.tail(N_HOLDOUT)
        holdout_indices.extend(holdout.index.tolist())
        holdout_rows.append(holdout)

    holdout_df = pd.concat(holdout_rows, ignore_index=False)
    n_holdout_users = holdout_df["userId"].nunique()
    print(f"  Held out {len(holdout_df):,} interactions from {n_holdout_users} users")

    # Create train ratings (everything minus held-out)
    train_ratings = ratings.drop(index=holdout_indices)
    print(f"  Train ratings: {len(train_ratings):,} rows")


    train_keys = set(zip(train_ratings["userId"], train_ratings["movieId"]))
    holdout_keys = set(zip(holdout_df["userId"], holdout_df["movieId"]))
    overlap = train_keys & holdout_keys
    assert len(overlap) == 0, f"DATA LEAK: {len(overlap)} overlapping interactions!"

    # Save
    train_ratings.to_csv(paths["train_ratings_csv"], index=False)
    holdout_df[["userId", "movieId", "rating", "timestamp"]].to_csv(
        paths["test_holdout_csv"], index=False
    )

    # Activity distribution of test users
    dist = {}
    for label in labels:
        count = sum(1 for uid in holdout_df["userId"].unique()
                    if user_bins.get(uid) == label)
        dist[label] = count
        print(f"    {label}: {count} test users")

    # Save metadata
    meta = {
        "n_test_users": int(n_holdout_users),
        "n_holdout_per_user": N_HOLDOUT,
        "holdout_threshold": HOLDOUT_THRESHOLD,
        "min_user_ratings": MIN_USER_RATINGS,
        "total_held_out": int(len(holdout_df)),
        "total_train_ratings": int(len(train_ratings)),
        "total_original_ratings": int(len(ratings)),
        "eval_seed": EVAL_SEED,
        "activity_distribution": dist,
    }
    paths["eval_split_meta"].write_text(json.dumps(meta, indent=2), encoding="utf-8")
    print(f"  Saved: {paths['train_ratings_csv'].name} ({len(train_ratings):,} rows)")
    print(f"  Saved: {paths['test_holdout_csv'].name} ({len(holdout_df):,} rows)")
    print(f"  Saved: {paths['eval_split_meta'].name}")

    return meta



In [8]:
# DATA CONVERSION

def convert_movielens_to_recbole(paths: dict) -> tuple[dict, dict]:
    """Convert MovieLens train_ratings.csv to RecBole .inter format."""
    print("Converting MovieLens to RecBole .inter format")
    print(f"{'─'*60}")

    # Use train_ratings.csv (post-split) instead of full ratings.csv
    source_csv = paths["train_ratings_csv"]
    if not source_csv.exists():
        print(f"  train_ratings.csv not found, falling back to full ratings.csv")
        source_csv = paths["ratings_csv"]
    assert source_csv.exists(), f"Missing: {source_csv}"

    # Load ratings
    dtypes = {"userId": "int32", "movieId": "int32", "rating": "float32", "timestamp": "int32"}
    ratings = pd.read_csv(source_csv, dtype=dtypes)
    print(f"  Loaded {len(ratings):,} ratings from {source_csv.name}")

    # Convert to implicit: keep only positive interactions
    ratings = ratings[ratings["rating"] >= RATING_THRESHOLD].copy()
    print(f"  After threshold ({RATING_THRESHOLD}): {len(ratings):,} positive interactions")

    # Build contiguous ID mappings for RecBole efficiency
    unique_users = sorted(ratings["userId"].unique())
    unique_items = sorted(ratings["movieId"].unique())

    user_id_map = {orig: idx for idx, orig in enumerate(unique_users)}
    item_id_map = {orig: idx for idx, orig in enumerate(unique_items)}

    # Map to contiguous IDs
    ratings["user_id"] = ratings["userId"].map(user_id_map)
    ratings["item_id"] = ratings["movieId"].map(item_id_map)

    print(f"  Users: {len(unique_users):,}  |  Items: {len(unique_items):,}")
    print(f"  Density: {len(ratings) / (len(unique_users) * len(unique_items)) * 100:.4f}%")

    # Sort by timestamp for temporal split
    if TEMPORAL_SPLIT:
        ratings = ratings.sort_values("timestamp")

    # Write .inter file
    inter_df = ratings[["user_id", "item_id", "rating", "timestamp"]].copy()
    inter_df.columns = ["user_id:token", "item_id:token", "rating:float", "timestamp:float"]

    inter_df.to_csv(paths["inter_file"], sep="\t", index=False)
    print(f"Saved: {paths['inter_file']} ({len(inter_df):,} rows)")

    # Save ID mappings (store as str keys for JSON)
    user_map_save = {str(k): v for k, v in user_id_map.items()}
    item_map_save = {str(k): v for k, v in item_id_map.items()}
    paths["user_id_map"].write_text(json.dumps(user_map_save), encoding="utf-8")
    paths["item_id_map"].write_text(json.dumps(item_map_save), encoding="utf-8")
    print(f"Saved ID maps: {paths['user_id_map'].name}, {paths['item_id_map'].name}")

    return user_id_map, item_id_map

In [ ]:
# RECBOLE CONFIG

def build_recbole_config(paths: dict) -> dict:
    """Build RecBole parameter dict for XSimGCL training."""
    config = {
        # Model
        "model": "XSimGCL",
        "dataset": "cinematch",
        "data_path": str(paths["recbole_data"].parent.resolve()),

        # Model hyperparams
        "embedding_size": EMBEDDING_SIZE,
        "n_layers": N_LAYERS,
        "cl_rate": CL_RATE,
        "noise_eps": NOISE_EPS,
        "reg_weight": REG_WEIGHT,

        # Data
        "USER_ID_FIELD": "user_id",
        "ITEM_ID_FIELD": "item_id",
        "RATING_FIELD": "rating",
        "TIME_FIELD": "timestamp",
        "load_col": {
            "inter": ["user_id", "item_id", "rating", "timestamp"],
        },
        "threshold": {"rating": RATING_THRESHOLD},

        # Eval (RecBole internal val/test from train data)
        "eval_args": {
            "split":    {"RS": SPLIT_RATIO},
            "group_by": "user",
            "order":    "TO",
            "mode":     "uni100",
        },
        "metrics":      ["Recall", "NDCG", "MRR"],
        "topk":         [10, 20, 50],
        "valid_metric": "NDCG@20",

        # Training
        "learning_rate": LEARNING_RATE,
        "train_batch_size": TRAIN_BATCH,
        "eval_batch_size": EVAL_BATCH,
        "epochs": EPOCHS,
        "stopping_step": EARLY_STOP,
        "eval_step": 3, 
        "weight_decay": 0.0,

        # GPU
        "gpu_id": 0 if torch.cuda.is_available() else -1,
        "use_gpu": torch.cuda.is_available(),
        "enable_amp": True,
        "enable_scaler": True,
        "mixed_precision": True,

        # Misc
        "seed": 42,
        "reproducibility": True,
        "checkpoint_dir": str(paths["model_dir"] / "checkpoints"),
        "show_progress": True,
        "log_wandb": False,
    }

    import yaml
    with open(paths["config_yaml"], "w") as f:
        yaml.dump(config, f, default_flow_style=False, allow_unicode=True)
    print(f"  Config saved: {paths['config_yaml']}")

    return config


In [ ]:

def train_xsimgcl(paths: dict, config: dict):
    """Train XSimGCL via RecBole-GNN and export embeddings."""
    import torch
    from tqdm.auto import tqdm
    import tqdm as tqdm_module
    tqdm_module.tqdm = tqdm
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    print("Training XSimGCL (RecBole-GNN)")
    print(f"{'─'*60}")

    try:
        from recbole_gnn.quick_start import run_recbole_gnn
        from recbole.utils import get_model, init_logger
        from recbole.config import Config
        from recbole.data import create_dataset, data_preparation
    except ImportError as e:
        print("\nRecBole-GNN not found or import failed.")
        raise

    # Train
    t0 = time.time()
    result = run_recbole_gnn(
        model="XSimGCL",
        dataset="cinematch",
        config_file_list=[str(paths["config_yaml"])],
        config_dict=config,
    )

    train_time = time.time() - t0
    print(f"\n  Training completed in {train_time:.1f}s")
    print(f"  Best valid metric: {result.get('best_valid_score', 'N/A')}")
    print(f"  Test results: {result.get('test_result', 'N/A')}")

    return result, train_time

In [11]:

def export_embeddings(paths: dict, config: dict, result: dict):
    """Load the best checkpoint and export user/item embedding matrices."""
    print("  Exporting embeddings from best checkpoint")
    print(f"{'─'*60}")

    from recbole.config import Config
    from recbole_gnn.utils import create_dataset, data_preparation
    import recbole.config.configurator as configurator
    import recbole.utils.utils as rb_utils
    from recbole_gnn.model.general_recommender.xsimgcl import XSimGCL

    _original_get_model = rb_utils.get_model
    def _mock_get_model(model_name):
        if model_name.lower() == 'xsimgcl':
            return XSimGCL
        return _original_get_model(model_name)

    rb_utils.get_model = _mock_get_model
    configurator.get_model = _mock_get_model

    checkpoint_path = result.get("best_valid_checkpoint")
    if not checkpoint_path:
        ckpt_dir = Path(config["checkpoint_dir"])
        checkpoints = sorted(ckpt_dir.glob("*.pth"), key=lambda p: p.stat().st_mtime)
        checkpoint_path = checkpoints[-1] if checkpoints else None

    assert checkpoint_path, "No checkpoint found"
    checkpoint_path = Path(checkpoint_path)
    print(f"  Checkpoint: {checkpoint_path}")

    rb_config = Config(model="XSimGCL", dataset="cinematch", config_dict=config)
    dataset = create_dataset(rb_config)
    _, _, test_data = data_preparation(rb_config, dataset)

    model = XSimGCL(rb_config, dataset)
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    model.load_state_dict(checkpoint["state_dict"])
    model.eval()

    print(f"  Users: {dataset.user_num:,}  |  Items: {dataset.item_num:,}")

    with torch.no_grad():
        user_emb = model.user_embedding.weight.data.cpu().numpy()
        item_emb = model.item_embedding.weight.data.cpu().numpy()

    print(f"  user_emb: {user_emb.shape}  |  item_emb: {item_emb.shape}")

    np.save(paths["user_emb"], user_emb)
    np.save(paths["item_emb"], item_emb)
    print(f"  Saved: {paths['user_emb'].name} and {paths['item_emb'].name}")

    return user_emb, item_emb


In [ ]:
# DEMOGRAPHIC CLUSTERS

def build_demographic_clusters(
    user_emb_path, user_id_map_path, demo_profiles_path,
    output_clusters_path, output_map_path,
) -> dict[str, np.ndarray]:
    """Build demographic cluster centroids from website users."""
    print("Building demographic cluster centroids")
    print(f"{'─'*60}")

    user_emb = np.load(user_emb_path)
    with open(user_id_map_path, "r") as f:
        user_id_map = json.load(f)

    demos = pd.read_csv(demo_profiles_path, dtype=str)
    required_cols = {"user_id", "age_group", "gender", "region"}
    assert required_cols.issubset(demos.columns)

    from collections import defaultdict
    cluster_vecs: dict[str, list[np.ndarray]] = defaultdict(list)

    matched, skipped = 0, 0
    for _, row in demos.iterrows():
        uid_str = str(row["user_id"])
        idx = user_id_map.get(uid_str)
        if idx is None or idx >= len(user_emb):
            skipped += 1
            continue
        matched += 1
        emb = user_emb[idx]
        profile = UserProfile(
            age_group=row.get("age_group", "undisclosed"),
            gender=row.get("gender", "undisclosed"),
            region=row.get("region", "Other"),
        )
        for key in profile.cluster_keys():
            cluster_vecs[str(key)].append(emb)

    print(f"  Matched {matched:,} users to embeddings (skipped {skipped:,})")

    centroids: dict[str, np.ndarray] = {}
    for key_str, vecs in cluster_vecs.items():
        if len(vecs) >= MIN_CLUSTER_SIZE:
            centroids[key_str] = np.mean(vecs, axis=0).astype(np.float32)

    print(f"  Clusters formed: {len(centroids)} (min size = {MIN_CLUSTER_SIZE})")

    if centroids:
        keys = list(centroids.keys())
        matrix = np.stack([centroids[k] for k in keys])
        np.save(output_clusters_path, matrix)
        Path(output_map_path).write_text(json.dumps(keys), encoding="utf-8")
        print(f"  Saved: {Path(output_clusters_path).name} ({matrix.shape})")
    else:
        print("  No clusters met minimum size; skipping save.")

    return centroids


In [13]:

def _lookup_demographic_centroid(profile, demo_clusters_path, demo_cluster_map_path):
    """Look up best matching demographic centroid for a user profile."""
    cluster_map_path = Path(demo_cluster_map_path)
    clusters_path = Path(demo_clusters_path)
    if not cluster_map_path.exists() or not clusters_path.exists():
        return None
    with open(cluster_map_path, "r") as f:
        cluster_keys = json.load(f)
    cluster_matrix = np.load(clusters_path)
    key_to_idx = {k: i for i, k in enumerate(cluster_keys)}
    for key in profile.cluster_keys():
        key_str = str(key)
        if key_str in key_to_idx:
            return cluster_matrix[key_to_idx[key_str]]
    return None


In [ ]:

# NEW USER SUPPORT

def register_new_user(
    user_interactions, item_emb_path, item_id_map_path,
    user_profile=None, demo_clusters_path=None, demo_cluster_map_path=None,
) -> np.ndarray:
    """Generate an embedding for a new website user."""
    item_emb = np.load(item_emb_path)
    with open(item_id_map_path, "r") as f:
        item_id_map = json.load(f)

    vecs = []
    for movie_id in user_interactions:
        idx = item_id_map.get(str(movie_id))
        if idx is not None and idx < len(item_emb):
            vecs.append(item_emb[idx])

    demo_centroid = None
    if user_profile and demo_clusters_path and demo_cluster_map_path:
        demo_centroid = _lookup_demographic_centroid(
            user_profile, demo_clusters_path, demo_cluster_map_path
        )

    if not vecs:
        if demo_centroid is not None:
            return demo_centroid
        return np.zeros(item_emb.shape[1], dtype=np.float32)

    interaction_emb = np.mean(vecs, axis=0).astype(np.float32)

    if demo_centroid is not None and len(vecs) < 5:
        alpha = DEMO_BLEND_WEIGHT * (1.0 - len(vecs) / 5.0)
        interaction_emb = ((1 - alpha) * interaction_emb + alpha * demo_centroid).astype(np.float32)

    return interaction_emb


In [15]:

def register_batch_new_users(
    user_interaction_map, item_emb_path, item_id_map_path,
    output_path=None, user_profiles=None,
    demo_clusters_path=None, demo_cluster_map_path=None,
) -> dict[str, np.ndarray]:
    """Batch register multiple new website users."""
    results = {}
    for uid, interactions in user_interaction_map.items():
        profile = user_profiles.get(uid) if user_profiles else None
        results[uid] = register_new_user(
            user_interactions=interactions,
            item_emb_path=item_emb_path,
            item_id_map_path=item_id_map_path,
            user_profile=profile,
            demo_clusters_path=demo_clusters_path,
            demo_cluster_map_path=demo_cluster_map_path,
        )

    if output_path:
        user_ids = list(results.keys())
        user_matrix = np.stack([results[uid] for uid in user_ids])
        np.save(output_path, user_matrix)
        map_path = Path(output_path).with_suffix(".json")
        map_path.write_text(json.dumps(user_ids), encoding="utf-8")
        print(f"New user embeddings: {user_matrix.shape} to {output_path}")

    return results

In [16]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:

def main():
    import time
    import pandas as pd
    import json
    t_start = time.time()
    paths = detect_paths()

    import numpy as np
    if not hasattr(np, 'float_'):
        np.float_ = np.float64
    if not hasattr(np, 'bool_'):
        np.bool_ = bool
    if not hasattr(np, 'int_'):
        np.int_ = np.int64
    if not hasattr(np, 'complex_'):
        np.complex_ = complex
    if not hasattr(np, 'object_'):
        np.object_ = object
    if not hasattr(np, 'unicode_'):
        np.unicode_ = np.str_

    #  Create eval split (removes held-out interactions)
    split_meta = create_eval_split(paths)

    #Convert train_ratings.csv → RecBole .inter format
    user_id_map, item_id_map = convert_movielens_to_recbole(paths)

    print("  Building RecBole config")
    print(f"{'─'*60}")
    config = build_recbole_config(paths)

    print(f"  XSimGCL params: emb={EMBEDDING_SIZE}, layers={N_LAYERS}, "
          f"cl_rate={CL_RATE}, noise_eps={NOISE_EPS}")

    # Train
    result, train_time = train_xsimgcl(paths, config)

    # Export embeddings
    user_emb, item_emb = export_embeddings(paths, config, result)

    # Save manifest
    manifest = {
        "model": "xsimgcl",
        "framework": "RecBole",
        "timestamp_utc": pd.Timestamp.utcnow().isoformat(),
        "dataset": {
            "source": str(paths["ratings_csv"]),
            "train_source": str(paths["train_ratings_csv"]),
            "users": len(user_id_map),
            "items": len(item_id_map),
            "rating_threshold": RATING_THRESHOLD,
            "temporal_split": TEMPORAL_SPLIT,
        },
        "eval_split": split_meta,
        "hyperparameters": {
            "embedding_size": EMBEDDING_SIZE,
            "n_layers": N_LAYERS,
            "cl_rate": CL_RATE,
            "noise_eps": NOISE_EPS,
            "reg_weight": REG_WEIGHT,
            "learning_rate": LEARNING_RATE,
            "epochs": EPOCHS,
            "early_stop": EARLY_STOP,
        },
        "results": {
            "best_valid_score": str(result.get("best_valid_score", "N/A")),
            "test_result": str(result.get("test_result", "N/A")),
            "train_time_seconds": round(train_time, 2),
        },
        "outputs": {
            "user_embedding_shape": list(user_emb.shape) if user_emb is not None else None,
            "item_embedding_shape": list(item_emb.shape) if item_emb is not None else None,
        },
    }
    paths["train_manifest"].write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print(f"\n Manifest saved: {paths['train_manifest']}")

    total = time.time() - t_start
    print(f"  XSimGCL DONE — Total time: {total:.1f}s")

if __name__ == "__main__":
    main()


Runtime: HPC
Creating evaluation split
────────────────────────────────────────────────────────────
  Loaded 32,000,204 total ratings
  Eligible users (≥20 ratings): 200,948
  Sampled 1000 test users (stratified by activity)
  Held out 9,790 interactions from 979 users
  Train ratings: 31,990,414 rows
    5-50: 183 test users
    50-100: 196 test users
    100-200: 200 test users
    200-500: 200 test users
    500+: 200 test users
  Saved: train_ratings.csv (31,990,414 rows)
  Saved: test_holdout.csv (9,790 rows)
  Saved: eval_split_meta.json
Converting MovieLens to RecBole .inter format
────────────────────────────────────────────────────────────
  Loaded 31,990,414 ratings from train_ratings.csv
  After threshold (3.5): 20,218,546 positive interactions
  Users: 200,806  |  Items: 65,022
  Density: 0.1549%
Saved: /blue/egn6933/nagabhairava.r/outputs/xsimgcl/dataset/cinematch/cinematch.inter (20,218,546 rows)
Saved ID maps: user_id_map.json, item_id_map.json
  Building RecBole config


23 Mar 18:58    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 42
state = INFO
reproducibility = True
data_path = /blue/egn6933/nagabhairava.r/outputs/xsimgcl/dataset/cinematch
checkpoint_dir = /blue/egn6933/nagabhairava.r/outputs/xsimgcl/checkpoints
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 50
train_batch_size = 262144
learner = adam
learning_rate = 0.001
train_neg_sample_args = {'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}
eval_step = 3
stopping_step = 5
clip_grad_norm = None
weight_decay = 0.0
loss_decimal_place = 4

Evaluation Hyper Parameters:
eval_args = {'split': {'RS': [0.8, 0.1, 0.1]}, 'group_by': 'user', 'order': 'TO', 'mode': 'uni100'}
repeatable = False
metrics = ['Recall', 'NDCG', 'MRR']
topk = [10, 20, 50]
valid_metric = NDCG@20
valid_metric_bigger = True
eval_batch_size 

Train     0:   0%|                                                           | 0/63 [00:00<?, ?it/s…

23 Mar 19:02    INFO  epoch 0 training [time: 107.82s, train_loss1: 40.6645, train_loss2: 0.0007, train_loss3: 89.2681]


Train     1:   0%|                                                           | 0/63 [00:00<?, ?it/s…

...

Evaluate   :   0%|                                                           | 0/16 [00:00<?, ?it/s…

23 Mar 20:53    INFO  best valid : OrderedDict({'recall@10': 0.751, 'recall@20': 0.85, 'recall@50': 0.9309, 'ndcg@10': 0.766, 'ndcg@20': 0.7901, 'ndcg@50': 0.8201, 'mrr@10': 0.8318, 'mrr@20': 0.8322, 'mrr@50': 0.8322})
23 Mar 20:53    INFO  test result: OrderedDict({'recall@10': 0.7277, 'recall@20': 0.8298, 'recall@50': 0.9162, 'ndcg@10': 0.7299, 'ndcg@20': 0.7564, 'ndcg@50': 0.7892, 'mrr@10': 0.8045, 'mrr@20': 0.805, 'mrr@50': 0.8051})



  Training completed in 6891.6s
  Best valid metric: 0.7901
  Test results: OrderedDict({'recall@10': 0.7277, 'recall@20': 0.8298, 'recall@50': 0.9162, 'ndcg@10': 0.7299, 'ndcg@20': 0.7564, 'ndcg@50': 0.7892, 'mrr@10': 0.8045, 'mrr@20': 0.805, 'mrr@50': 0.8051})
  Exporting embeddings from best checkpoint
────────────────────────────────────────────────────────────
  Checkpoint: /blue/egn6933/nagabhairava.r/outputs/xsimgcl/checkpoints/XSimGCL-Mar-23-2026_19-00-16.pth


23 Mar 20:55    INFO  [Training]: train_batch_size = [262144] train_neg_sample_args: [{'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}]
23 Mar 20:55    INFO  [Evaluation]: eval_batch_size = [70000000] eval_args: [{'split': {'RS': [0.8, 0.1, 0.1]}, 'group_by': 'user', 'order': 'TO', 'mode': 'uni100'}]


  Users: 200,807  |  Items: 65,023
  user_emb: (200807, 512)  |  item_emb: (65023, 512)
  Saved: user_embeddings.npy and item_embeddings.npy

 Manifest saved: /blue/egn6933/nagabhairava.r/outputs/xsimgcl/train_manifest.json
  XSimGCL DONE — Total time: 7066.9s


In [18]:
# torch.cuda.empty_cache()